# Etapa 1 - EDA + Baselines
\n
Notebook inicial para explorar dados de churn e rodar baselines com MLflow.

In [17]:
from pathlib import Path
import pandas as pd

DATA_PATH = Path("../data/Telco_customer_churn.xlsx")
TARGET_COL = "Churn Value"

In [18]:
if DATA_PATH.suffix.lower() in {".xlsx", ".xls"}:
    df = pd.read_excel(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(7043, 33)


,CustomerID,Count,Country,State,City,Zip Code,Lat Long,Latitude,Longitude,Gender,...,Contract,Paperless Billing,Payment Method,Monthly Charges,Total Charges,Churn Label,Churn Value,Churn Score,CLTV,Churn Reason
0,3668-QPYBK,1,United States,California,Los Angeles,90003,"33.964131, -118.272783",33.964131,-118.272783,Male,...,Month-to-month,Yes,Mailed check,53.85,108.15,Yes,1,86,3239,Competitor made better offer
1,9237-HQITU,1,United States,California,Los Angeles,90005,"34.059281, -118.30742",34.059281,-118.307420,Female,...,Month-to-month,Yes,Electronic check,70.70,151.65,Yes,1,67,2701,Moved
2,9305-CDSKC,1,United States,California,Los Angeles,90006,"34.048013, -118.293953",34.048013,-118.293953,Female,...,Month-to-month,Yes,Electronic check,99.65,820.5,Yes,1,86,5372,Moved
3,7892-POOKP,1,United States,California,Los Angeles,90010,"34.062125, -118.315709",34.062125,-118.315709,Female,...,Month-to-month,Yes,Electronic check,104.80,3046.05,Yes,1,84,5003,Moved
4,0280-XJGEX,1,United States,California,Los Angeles,90015,"34.039224, -118.266293",34.039224,-118.266293,Male,...,Month-to-month,Yes,Bank transfer (automatic),103.70,5036.3,Yes,1,89,5340,Competitor had better devices


In [19]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 33 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   CustomerID         7043 non-null   object 
 1   Count              7043 non-null   int64  
 2   Country            7043 non-null   object 
 3   State              7043 non-null   object 
 4   City               7043 non-null   object 
 5   Zip Code           7043 non-null   int64  
 6   Lat Long           7043 non-null   object 
 7   Latitude           7043 non-null   float64
 8   Longitude          7043 non-null   float64
 9   Gender             7043 non-null   object 
 10  Senior Citizen     7043 non-null   object 
 11  Partner            7043 non-null   object 
 12  Dependents         7043 non-null   object 
 13  Tenure Months      7043 non-null   int64  
 14  Phone Service      7043 non-null   object 
 15  Multiple Lines     7043 non-null   object 
 16  Internet Service   7043 

In [20]:
missing = df.isna().sum().sort_values(ascending=False)
missing[missing > 0].head(20)

Churn Reason    5174
dtype: int64

In [21]:
df[TARGET_COL].value_counts(normalize=True)

Churn Value
0    0.73463
1    0.26537
Name: proportion, dtype: float64

## EDA - Analise Exploratoria de Dados

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# --- 1. Distribuicao do target ---
fig, ax = plt.subplots(figsize=(5, 4))
counts = df[TARGET_COL].value_counts()
ax.bar(["Nao Churn (0)", "Churn (1)"], counts.values, color=["steelblue", "tomato"])
ax.set_title("Distribuicao de Churn")
ax.set_ylabel("Numero de clientes")
for i, v in enumerate(counts.values):
    ax.text(i, v + 30, f"{v} ({v/len(df)*100:.1f}%)", ha="center")
plt.tight_layout()
plt.savefig("../docs/eda_target_distribution.png", dpi=120)
plt.show()

In [ ]:
# --- 2. Variaveis numericas: distribuicao por churn ---
num_cols = ["Tenure Months", "Monthly Charges", "Total Charges", "CLTV", "Churn Score"]
fig, axes = plt.subplots(1, len(num_cols), figsize=(20, 4))
for ax, col in zip(axes, num_cols):
    for val, label, color in [(0, "Nao Churn", "steelblue"), (1, "Churn", "tomato")]:
        subset = df[df[TARGET_COL] == val][col].dropna()
        ax.hist(subset, bins=30, alpha=0.6, label=label, color=color, density=True)
    ax.set_title(col)
    ax.legend(fontsize=8)
plt.suptitle("Distribuicao de variaveis numericas por Churn", y=1.02)
plt.tight_layout()
plt.savefig("../docs/eda_numeric_distributions.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- 3. Variaveis categoricas: taxa de churn por categoria ---
cat_cols = ["Contract", "Payment Method", "Internet Service", "Tech Support", "Senior Citizen"]
fig, axes = plt.subplots(1, len(cat_cols), figsize=(22, 5))
for ax, col in zip(axes, cat_cols):
    churn_rate = df.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
    churn_rate.plot(kind="bar", ax=ax, color="tomato", edgecolor="white")
    ax.set_title(f"Churn rate por\n{col}", fontsize=10)
    ax.set_ylabel("Taxa de churn")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=30, labelsize=8)
    ax.set_ylim(0, 1)
plt.suptitle("Taxa de churn por variavel categorica", y=1.02)
plt.tight_layout()
plt.savefig("../docs/eda_categorical_churn_rates.png", dpi=120, bbox_inches="tight")
plt.show()

In [ ]:
# --- 4. Boxplots: Monthly Charges e Tenure Months por churn ---
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
for ax, col in zip(axes, ["Monthly Charges", "Tenure Months"]):
    groups = [df[df[TARGET_COL] == v][col].dropna() for v in [0, 1]]
    ax.boxplot(groups, labels=["Nao Churn", "Churn"], patch_artist=True,
               boxprops=dict(facecolor="lightblue"),
               medianprops=dict(color="red", linewidth=2))
    ax.set_title(col)
    ax.set_ylabel(col)
plt.suptitle("Boxplots por status de Churn")
plt.tight_layout()
plt.savefig("../docs/eda_boxplots.png", dpi=120)
plt.show()

In [ ]:
# --- 5. Correlacao entre variaveis numericas ---
num_df = df[["Tenure Months", "Monthly Charges", "Total Charges", "CLTV", "Churn Score", TARGET_COL]].copy()
num_df[TARGET_COL] = num_df[TARGET_COL].astype(float)
corr = num_df.corr()

fig, ax = plt.subplots(figsize=(7, 6))
mask = [[i < j for j in range(len(corr.columns))] for i in range(len(corr.columns))]
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", ax=ax,
            linewidths=0.5, vmin=-1, vmax=1)
ax.set_title("Matriz de correlacao - variaveis numericas")
plt.tight_layout()
plt.savefig("../docs/eda_correlation_matrix.png", dpi=120)
plt.show()

In [22]:
import sys
sys.path.append("../src")

from churn_stage1.stage1_baselines import run_stage1

In [23]:
metrics_df, run_id = run_stage1(
    data_path=DATA_PATH,
    target_col=TARGET_COL,
    experiment_name="tech_challenge_stage1",
    test_size=0.2,
    random_state=42,
)
print(f"run_id (logistic_regression): {run_id}")
metrics_df

2026/04/13 20:12:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 20:12:50 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'churn_dummy_most_frequent' already exists. Creating a new version of this model...
Created version '3' of model 'churn_dummy_most_frequent'.
2026/04/13 20:12:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/13 20:12:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute ar

run_id (logistic_regression): 9abdbe9ebe7548d4839ee521a45826ff


Registered model 'churn_logistic_regression' already exists. Creating a new version of this model...
Created version '3' of model 'churn_logistic_regression'.


,model,auc_roc,pr_auc,f1
1,logistic_regression,0.829027,0.607406,0.611791
0,dummy_most_frequent,0.500000,0.265436,0.000000


In [24]:
from churn_stage1.predict import predict_churn

# Substitua run_id pelo valor impresso na celula anterior
model_uri = f"runs:/{run_id}/model"

pred = predict_churn(
    model_uri=model_uri,
    data_path=DATA_PATH,
    output_path=Path("../models/churn_predictions.csv"),
    target_col=TARGET_COL,
    threshold=0.5,
)
pred.head(10)

,customer_id,churn_score,churn_pred
598,7180-PISOG,0.991932,1
703,1569-TTNYJ,0.985829,1
1474,7228-OMTPN,0.985643,1
1009,2656-TABEH,0.984040,1
618,1846-XWOQN,0.983564,1
1823,7384-GHBPI,0.983287,1
1163,1455-UGQVH,0.982557,1
178,4952-YSOGZ,0.982449,1
1643,2446-BEGGB,0.981276,1
1226,9965-YOKZB,0.980018,1


## Proximos passos
- Completar ML Canvas em `docs/ml_canvas_template.md`\n
- Refinar EDA com graficos e hipoteses de negocio\n
- Capturar screenshots/metricas do MLflow para o README